# TP2 - Reconhecimento de Dígitos Manuscritos usando PCA
## Álgebra Linear Computacional - UFMG

- Clara Garcia Tavares, 2022103380;
- Náthally Fernandes de Brito Oliveira, 2022035431.

## Importações

In [ ]:
import sklearn
from sklearn import model_selection
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels
import warnings
warnings.filterwarnings('ignore')

: 

## Função para plotar matriz de confusão

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes,
                        normalize=False,
                        title=None,
                        cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    # Only use the labels that appear in the data
    classes = np.array(classes)
    classes = classes[unique_labels(y_true, y_pred)]
    fig, ax = plt.subplots(figsize = (8,8))
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    # We want to show all ticks...
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           # ... and label them with the respective list entries
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')
    ax.set_ylim(cm.shape[0] -1 + 0.5, -0.5)
    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
            rotation_mode="anchor")
    # Loop over data dimensions and create text annotations.
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    plt.show()
    return cm

## PARTE 1 - Carregando e preparando os dados

In [ ]:
# Carrega o dataset de dígitos
digits = load_digits()
print('Shape do dataset:', digits.data.shape)
print('Formato: (número de amostras, número de pixels por amostra)')

## Visualizando alguns exemplos de dígitos

In [ ]:
# Vamos visualizar alguns exemplos
indices = [0, 300, 500, 800, 1000, 1200, 1400, 1600]
figs, axs = plt.subplots(2, 4, figsize=(10, 5))
i = 0
for ind in indices:
    axs[int(i/4)][i%4].matshow(digits.images[ind], cmap='gray')
    axs[int(i/4)][i%4].set_title('Classe %s' % (digits.target[ind]))
    axs[int(i/4)][i%4].axis('off')
    i += 1
plt.tight_layout()
plt.show()

## PARTE 1 - Separando dados em treino e teste

In [ ]:
# split 90/10
X = digits.data
y = digits.target

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.1, random_state=1
)

# Confirmando que temos todas as classes no conjunto de teste
y_aux = np.argsort(y_test)
y_test = y_test[y_aux]
X_test = X_test[y_aux]

items, counts = np.unique(y_test, return_counts=True)
print('Classes presentes no conjunto de teste:', items)
print('Número de amostras por classe:', counts)
print(f'\nTotal de amostras de treino: {X_train.shape[0]}')
print(f'Total de amostras de teste: {X_test.shape[0]}')

## PARTE 2 - Aplicando PCA

O método PCA reduz a dimensionalidade dos dados enquanto preserva a maior parte da variância, de modo que cada dígito pode ser representado como uma combinação linear de componentes principais.

In [ ]:
# Primeiro, vamos calcular a média de cada pixel no conjunto de treino
mat_media = np.mean(X_train, axis=0)

X_train_centrada = X_train - mat_media

print('Dados centrados. Shape:', X_train_centrada.shape)
print('Média dos dados centrados:', np.mean(X_train_centrada))

## PARTE 2 - Computando componentes principais para k = 5, 10, 15, 20

In [ ]:
# Dicionário para armazenar os modelos PCA para cada k
pca_models = {}
k_values = [5, 10, 15, 20]

# Para cada valor de k, treinar um modelo PCA
for k in k_values:
    pca = PCA(n_components=k, random_state=42)
    pca.fit(X_train_centrada)
    pca_models[k] = pca
    
    # Imprimir informações sobre a variância explicada
    variancia_total = np.sum(pca.explained_variance_ratio_)
    print(f'\nk = {k}:')
    print(f'  Variância explicada: {variancia_total:.4f} ({variancia_total*100:.2f}%)')
    print(f'  Número de componentes: {pca.n_components_}')

## PARTE 2 - Visualizando as autofaces

As autofaces são os autovetores da matriz de covariância, redimensionados para formar imagens.

In [ ]:
# Visualizar as primeiras autofaces para k=20
pca_20 = pca_models[20]
autofaces = pca_20.components_.reshape((20, 8, 8))

fig, axes = plt.subplots(4, 5, figsize=(12, 10))
axes = axes.ravel()

for i in range(20):
    axes[i].imshow(autofaces[i], cmap='gray')
    axes[i].set_title(f'Autovetor {i+1}\n(Variância: {pca_20.explained_variance_ratio_[i]:.3f})')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## PARTE 3 - Classificação de dígitos usando PCA

**Sobre kostas_decomposition:** Neste trabalho, usamos a variável kostas_decomposition para armazenar
os coeficientes da decomposição dos dados no espaço dos componentes principais. Esta variável contém
as projeções dos dígitos centrados nos primeiros k autovetores da matriz de covariância.
Cada linha de kostas_decomposition representa um dígito, e cada coluna representa o coeficiente
de um componente principal. A classificação é feita encontrando o dígito de treino mais próximo
(em termos de distância Euclidiana no espaço dos componentes) para cada dígito de teste.

In [ ]:
predictions = {}
coeficientes_treino = {}
coeficientes_teste = {}

for k in k_values:
    print(f'\n=== Processando k = {k} ===')
    pca = pca_models[k]
    
    # Transformar dados de treino (kostas_decomposition para treino)
    kostas_decomposition_train = pca.transform(X_train_centrada)
    coeficientes_treino[k] = kostas_decomposition_train
    
    X_test_centrada = X_test - mat_media
    kostas_decomposition_test = pca.transform(X_test_centrada)
    coeficientes_teste[k] = kostas_decomposition_test
    
    print(f'Shape kostas_decomposition (treino): {kostas_decomposition_train.shape}')
    print(f'Shape kostas_decomposition (teste): {kostas_decomposition_test.shape}')
    
    coeficientes_media = []
    for digit_class in range(10):
        indices_classe = np.where(y_train == digit_class)[0]
        media_classe = np.mean(kostas_decomposition_train[indices_classe, :], axis=0)
        coeficientes_media.append(media_classe)
    
    coeficientes_media = np.array(coeficientes_media)
    
    # Classificar: encontrar a classe mais próxima para cada amostra de teste
    y_pred = []
    for i in range(len(X_test)):
        # Calcular distância para cada classe média
        distancias = np.sum((coeficientes_media - kostas_decomposition_test[i, :]) ** 2, axis=1)
        classe_predita = np.argmin(distancias)
        y_pred.append(classe_predita)
    
    predictions[k] = np.array(y_pred)
    
    # Calcular acurácia
    acuracia = np.mean(y_pred == y_test)
    print(f'Acurácia: {acuracia:.4f} ({acuracia*100:.2f}%)')

## PARTE 4 - Gerando matrizes de confusão

In [ ]:
confusion_matrices = {}

for k in k_values:
    print(f'\n=== Matriz de Confusão para k = {k} ===')
    y_pred = predictions[k]
    
    # Gerar matriz de confusão
    cm = plot_confusion_matrix(y_test, y_pred, classes=np.arange(10),
                               title=f'Matriz de Confusão - k = {k}')
    confusion_matrices[k] = cm
    print('\nMatriz de Confusão:')
    print(cm)

## PARTE 5 - Calculando Precisão Média e Recall Médio

In [ ]:
resultados = []

print('\n' + '='*70)
print('RESULTADOS FINAIS')
print('='*70)

for k in k_values:
    cm = confusion_matrices[k]
    
    # Calcular somas das linhas (Ci+) e colunas (C+i)
    C_i_plus = np.sum(cm, axis=1)  # Soma por linha
    C_plus_i = np.sum(cm, axis=0)  # Soma por coluna
    
    precisoes = np.diag(cm) / C_i_plus
    precisao_media = np.mean(precisoes)
    
    recalls = np.diag(cm) / C_plus_i
    recall_medio = np.mean(recalls)
    
    acuracia = np.trace(cm) / np.sum(cm)
    
    resultados.append({
        'k': k,
        'precisao_media': precisao_media,
        'recall_medio': recall_medio,
        'acuracia': acuracia,
        'precisoes': precisoes,
        'recalls': recalls
    })
    
    print(f'\nk = {k}')
    print(f'-' * 40)
    print(f'Precisão Média:  {precisao_media:.6f}')
    print(f'Recall Médio:    {recall_medio:.6f}')
    print(f'Acurácia:        {acuracia:.6f}')
    print(f'\nPrecisão por classe:')
    for i in range(10):
        print(f'  Classe {i}: {precisoes[i]:.4f}')
    print(f'\nRecall por classe:')
    for i in range(10):
        print(f'  Classe {i}: {recalls[i]:.4f}')

print('\n' + '='*70)

## Comparação de desempenho entre valores de k

In [ ]:
print('\n' + '='*70)
print('RESUMO COMPARATIVO')
print('='*70)
print(f'{'k':^5} | {'Precisão Média':^16} | {'Recall Médio':^16} | {'Acurácia':^12}')
print('-' * 70)

for resultado in resultados:
    k = resultado['k']
    pm = resultado['precisao_media']
    rm = resultado['recall_medio']
    ac = resultado['acuracia']
    print(f'{k:^5} | {pm:^16.6f} | {rm:^16.6f} | {ac:^12.6f}')

print('='*70)

## Visualização do impacto de k no desempenho

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ks = [r['k'] for r in resultados]
precisoes_medias = [r['precisao_media'] for r in resultados]
recalls_medios = [r['recall_medio'] for r in resultados]
acuracias = [r['acuracia'] for r in resultados]

# Gráfico 1: Precisão Média
axes[0].plot(ks, precisoes_medias, 'o-', linewidth=2, markersize=8, color='blue')
axes[0].set_xlabel('Número de Componentes (k)', fontsize=12)
axes[0].set_ylabel('Precisão Média', fontsize=12)
axes[0].set_title('Precisão Média vs k', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(ks)

# Gráfico 2: Recall Médio
axes[1].plot(ks, recalls_medios, 's-', linewidth=2, markersize=8, color='green')
axes[1].set_xlabel('Número de Componentes (k)', fontsize=12)
axes[1].set_ylabel('Recall Médio', fontsize=12)
axes[1].set_title('Recall Médio vs k', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(ks)

# Gráfico 3: Acurácia
axes[2].plot(ks, acuracias, '^-', linewidth=2, markersize=8, color='red')
axes[2].set_xlabel('Número de Componentes (k)', fontsize=12)
axes[2].set_ylabel('Acurácia', fontsize=12)
axes[2].set_title('Acurácia vs k', fontsize=14)
axes[2].grid(True, alpha=0.3)
axes[2].set_xticks(ks)

plt.tight_layout()
plt.show()

## Análise

Visualizaão de como um dígito de teste é aproximado usando diferentes números de componentes principais.

In [ ]:
# Selecionar um dígito de teste aleatório
idx_teste = 50
digito_original = X_test[idx_teste].reshape(8, 8)
digito_verdadeiro = y_test[idx_teste]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

# Mostrar dígito original
axes[0].imshow(digito_original, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

# Mostrar aproximações para cada k
for idx, k in enumerate(k_values):
    pca = pca_models[k]
    
    # Projetar no espaço PCA
    kostas_decomp = pca.transform((X_test[idx_teste:idx_teste+1] - mat_media))
    
    # Reconstruir
    digito_reconstruido = pca.inverse_transform(kostas_decomp) + mat_media
    digito_reconstruido = digito_reconstruido.reshape(8, 8)
    
    axes[idx+1].imshow(digito_reconstruido, cmap='gray')
    axes[idx+1].set_title(f'k={k}')
    axes[idx+1].axis('off')

plt.suptitle(f'Dígito {digito_verdadeiro} - Reconstrução com diferentes k', fontsize=14)
plt.tight_layout()
plt.show()

## Conclusões

O uso do PCA mostrou que é possível reduzir drasticamente a dimensionalidade dos dados (de 64 para até 20 componentes) mantendo uma boa taxa de acerto na classificação. A variável kostas_decomposition guardou essas projeções, e as distâncias euclidianas no espaço reduzido foram suficientes para identificar os dígitos. Como esperado, k maiores dão resultados melhores, mas o custo computacional também sobe.

**Detalhamento:**

**Impacto de k**: Quanto maior o número de componentes principais (k), melhor é a reconstrução dos
   dígitos e, geralmente, melhor é o desempenho de classificação. Entretanto, há um custo computacional
   associado ao uso de mais componentes.

**Redução de dimensionalidade**: Os dígitos originais têm 64 dimensões (8x8 pixels). Usando apenas
   20 componentes principais, conseguimos capturar a maior parte da variância nos dados, permitindo
   uma classificação eficiente com redução significativa de complexidade.

**Interpretabilidade**: Ao contrário de métodos de caixa preta, o PCA oferece componentes interpretáveis
   que podem ser visualizados como "autofaces" para dígitos, revelando características importantes que
   diferenciam as classes.